In [1]:
import requests
import numpy as np
import casadi as ca
import matplotlib.pyplot as plt
import os
from IPython.display import clear_output
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import gpytorch
from common import get_data, to_tensor
import constant as const
from call_GP import train_GP, ExactGPModel, evaluate_gp, call_model, cal_pos

url = "http://127.0.0.1:80"
testcase = 'bestest_air'
testid = \
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
save_dir = 'results'
os.makedirs(save_dir, exist_ok=True)

print("done")

Using: cuda
done


In [2]:
testid = \
requests.post('{0}/testcases/{1}/select'.format(url,testcase)).json()['testid']
name = requests.get('{0}/name/{1}'.format(url, testid)).json()['payload']
print(name)

version = requests.get('{0}/version'.format(url)).json()['payload']
print(version)
inputs = requests.get('{0}/inputs/{1}'.format(url, testid)).json()['payload']
print('TEST CASE INPUTS ---------------------------------------------')
print(inputs.keys())

print('TEST CASE MEASUREMENTS ---------------------------------------')
measurements = requests.get('{0}/measurements/{1}'.format(url, testid)).json()['payload']

{'name': 'bestest_air'}
{'version': '0.8.0'}
TEST CASE INPUTS ---------------------------------------------
dict_keys(['con_oveTSetCoo_activate', 'con_oveTSetCoo_u', 'con_oveTSetHea_activate', 'con_oveTSetHea_u', 'fcu_oveFan_activate', 'fcu_oveFan_u', 'fcu_oveTSup_activate', 'fcu_oveTSup_u'])
TEST CASE MEASUREMENTS ---------------------------------------


In [3]:
step = const.step

data = pd.read_csv(f'processed_uniform_low_step_5m_data.csv')
data_test = data

data.head()

T_zone = (data['zon_reaTRooAir_y'].values - 273.15) / 30.0
T_sup  = (data['fcu_oveTSup_u'].values - 273.15) / 40.0
fan  = data['fcu_oveFan_u'].values
T_out = (data['zon_weaSta_reaWeaTDryBul_y'].values - 273.15) / 10.0


pos_T_zone = T_zone[1:]
cur_T_zone = T_zone[:-1]
cur_T_sup = T_sup[:-1]
cur_fan = fan[:-1]
cur_T_out = T_out[:-1]


def data_split(Ns, Ne, x1, x2, x3, x4, x5):
    return (to_tensor(x1[Ns:Ne]).view(-1,1), to_tensor(x2[Ns:Ne]).view(-1,1),
            to_tensor(x3[Ns:Ne]).view(-1,1), to_tensor(x4[Ns:Ne]).view(-1,1), to_tensor(x5[Ns:Ne]).view(-1,1))


Ns_tr, Ne_tr = const.Ns_tr, const.Ne_tr  
Ns_t, Ne_t = const.Ns_t, const.Ne_t 
cur_T_zone_tr, cur_T_sup_tr, cur_fan_tr, cur_T_out_tr, pos_T_zone_tr = data_split(Ns_tr, Ne_tr, cur_T_zone, cur_T_sup, cur_fan, cur_T_out, pos_T_zone)
cur_T_zone_t, cur_T_sup_t, cur_fan_t, cur_T_out_t, pos_T_zone_t = data_split(Ns_t, Ne_t, cur_T_zone, cur_T_sup, cur_fan, cur_T_out, pos_T_zone)

X_tr = torch.cat((cur_T_zone_tr, cur_T_sup_tr, cur_fan_tr, cur_T_out_tr), dim=1)
y_tr = pos_T_zone_tr.squeeze(-1)  
X_t = torch.cat((cur_T_zone_t, cur_T_sup_t, cur_fan_t, cur_T_out_t), dim=1)
y_t = pos_T_zone_t.squeeze(-1)

ramp = const.ramp

In [4]:
class MPC:
    def __init__(self, X_tr, y_tr, model_path=f'model_UCB_GP_{step}m.pth'):
        self.likelihood = gpytorch.likelihoods.GaussianLikelihood().to(device)
        self.gp_model = ExactGPModel(X_tr, y_tr, self.likelihood).to(device)
        
        state_dict = torch.load(model_path, map_location=device)
        self.gp_model.load_state_dict(state_dict)
        self.gp_model.eval()

    def cal_var(self, x0, u, x_out):
        if not isinstance(x0, torch.Tensor):
            x0 = torch.tensor(x0, dtype=torch.float32, device=device)
        if not isinstance(u, torch.Tensor):
            u = torch.tensor(u, dtype=torch.float32, device=device)
        if not isinstance(x_out, torch.Tensor):
            x_out = torch.tensor(x_out, dtype=torch.float32, device=device)
        x0 = x0.to(device)
        u = u.to(device)
        x_out = x_out.to(device)
        if x0.dim() == 1:
            x0 = x0.unsqueeze(-1)
        if x_out.dim() == 1:
            x_out = x_out.unsqueeze(-1)
        xu = torch.cat((x0, u, x_out), dim=1)
        with torch.no_grad(), gpytorch.settings.fast_pred_var():
            pred = self.gp_model(xu)
            scores = pred.mean + 10 * pred.variance
        return scores

    def eval_cost(self, x0, u, x_out):
        return self.cal_var(x0, u, x_out)

    def mpc_grid_search(self, x0, x_out, horizon=1, n_grid=20, u0_1=0.5, u0_2=0.5, plot=True):
        x0    = torch.tensor([[x0 / 30.0]], dtype=torch.float32, device=device)
        x_out = torch.tensor([[x_out]],     dtype=torch.float32, device=device)

        u_min_general = np.array([0.3, 0.])
        u_max_general = np.array([1.0, 1.0])

        u_min = np.clip(np.array([u0_1 - ramp, u0_2 - ramp]), u_min_general, u_max_general)
        u_max = np.clip(np.array([u0_1 + ramp, u0_2 + ramp]), u_min_general, u_max_general)

        u1_vals = torch.linspace(float(u_min[0]), float(u_max[0]), n_grid, device=device)
        u2_vals = torch.linspace(float(u_min[1]), float(u_max[1]), n_grid, device=device)

        U_grid = torch.cartesian_prod(u1_vals, u2_vals)
        N = U_grid.shape[0]
        X_cands = torch.cat([x0.expand(N, -1), U_grid, x_out.expand(N, -1)], dim=1)

        scores = self.eval_cost(X_cands[:, :1], X_cands[:, 1:3], X_cands[:, 3:])

        exclude = (U_grid[:, 0] == u0_1) | (U_grid[:, 1] == u0_2)
        scores_sel = scores.masked_fill(exclude, float('-inf'))
        ind    = int(torch.argmax(scores_sel))
        best_u = U_grid[ind]

        cost_grid = scores.reshape(n_grid, n_grid).cpu().numpy()

        if plot:
            U1, U2 = np.meshgrid(u1_vals.cpu().numpy(), u2_vals.cpu().numpy(), indexing="ij")
            plt.figure(figsize=(6,5))
            cp = plt.contourf(U1, U2, cost_grid, levels=30, cmap="viridis")
            plt.colorbar(cp, label="Cost (-variance)")
            plt.scatter(best_u[0].cpu(), best_u[1].cpu(), c="red", marker="x", s=100)
            plt.xlabel("u1 (supply)")
            plt.ylabel("u2 (airflow)")
            plt.title("Grid Search Cost Landscape")
            plt.show()

        u = {
        'fcu_oveTSup_u': best_u[0].item() * 40 + 273.15,
        'fcu_oveTSup_activate': 1,
        'fcu_oveFan_u': best_u[1].item(),
        'fcu_oveFan_activate': 1
        }
        return u

In [ ]:
y = requests.put('{0}/scenario/{1}'.format(url, testid), 
                 json={'time_period':'typical_heat_day',
                       'electricity_price':'dynamic'}).json()['payload']['time_period']
requests.put('{0}/step/{1}'.format(url, testid), json={'step':step*60})
start_time_days = y['time']/3600/24
train_GP(X_tr=X_tr, y_tr=y_tr.squeeze(-1), save_path=f'model_UCB_GP_{step}m.pth', ori=0)
evaluate_gp(X_tr, y_tr, X_t, y_t, save_path=f'model_UCB_GP_{step}m.pth')

RuntimeError: CUDA error: unknown error
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


: 

In [ ]:
last_change_day = start_time_days
counter = 0
u0_1 = 0.5
u0_2 = 0.5
u = {
        'fcu_oveTSup_u': u0_1 * 40 + 273.15,
        'fcu_oveTSup_activate': 1,
        'fcu_oveFan_u': u0_2,
        'fcu_oveFan_activate': 1
        }

x_ref = 25
simulation_time_days = y['time']/3600/24
print('Simulation time [elapsed days] = {:.2f}'.format((simulation_time_days - start_time_days)))
y = requests.post('{0}/advance/{1}'.format(url, testid), json=u).json()['payload']  


ls_rmse = [evaluate_gp(X_tr, y_tr, X_t, y_t, save_path=f'model_UCB_GP_{step}m.pth')]
X_tr_new = X_tr
y_tr_new = y_tr
num_count = const.num_count

In [ ]:
while y:
    x0 = y['zon_reaTRooAir_y']-273.15
    x_out = (y['zon_weaSta_reaWeaTDryBul_y']-273.15)/10.0
    mpc_controller = MPC(X_tr_new, y_tr_new, model_path=f'model_UCB_GP_{step}m.pth')
    u = mpc_controller.mpc_grid_search(x0=x0, x_out=x_out, horizon=1, u0_1=u0_1, u0_2=u0_2, n_grid=10, plot =False)

    print('-------------------------------------------------------------------')
    print('Step counter =', counter)
    print("Controller output:", u)

    y = requests.post('{0}/advance/{1}'.format(url, testid), json=u).json()['payload']  
    counter += 1


    cur = to_tensor(x0).view(-1,1)/30.0
    T_sup_add=  to_tensor((u['fcu_oveTSup_u']-273.15)/40.0).view(-1,1)
    fan_add = to_tensor(u['fcu_oveFan_u']).view(-1,1)
    pos_add = to_tensor(y['zon_reaTRooAir_y']-273.15).view(-1,1)/30.0

    T_out_add = to_tensor((y['zon_weaSta_reaWeaTDryBul_y']-273.15)/10.0).view(-1,1)

    X_tr_n = torch.cat((cur, T_sup_add, fan_add, T_out_add), dim=1)

    y_tr_n = pos_add.squeeze(-1)   

    X_tr_new = torch.cat((X_tr_new, X_tr_n), dim=0)
    y_tr_new = torch.cat((y_tr_new, y_tr_n), dim=0)
    
    if const.count_data == const.number_of_data:
        train_GP(X_tr=X_tr_new, y_tr=y_tr_new, save_path=f'model_UCB_GP_{step}m.pth', ori=1)
        print("=======retrain GP model=======")
        const.count_data = 0
    const.count_data += 1
    simulation_time_days = y['time']/3600/24
    print('Simulation time [elapsed days] = {:.2f}'.format((simulation_time_days - start_time_days)))

    u0_1 = (u['fcu_oveTSup_u']-273.15)/40.0
    u0_2 = u['fcu_oveFan_u']

    rmse = evaluate_gp(X_tr_new, y_tr_new, X_t, y_t, save_path=f'model_UCB_GP_{step}m.pth')

    ls_rmse.append(rmse)
    if counter > num_count:
        break

In [ ]:
def get_and_plot_results(testid, start_time, final_time):
    points = ['zon_reaTRooAir_y', 'fcu_oveTSup_u', 'fcu_oveFan_u', 'zon_weaSta_reaWeaTDryBul_y']
    args = {
        'point_names': points,
        'start_time': start_time, 
        'final_time': final_time
    }
    
    response = requests.put('{0}/results/{1}'.format(url, testid), json=args).json()
    
    if 'payload' not in response:
        print("Error: Could not retrieve data. Check if testid is still active.")
        return None
        
    df_res = pd.DataFrame(data=response['payload'])
    
    df_res = df_res.set_index('time')
    x_time = df_res.index / 3600. / 24.
    x_time = x_time - (start_time / 3600. / 24.)
    df_res['time'] = x_time
    

    plt.close('all')
    fig, axs = plt.subplots(4, 1, sharex=True, figsize=(10, 10))
    
    axs[0].plot(x_time, df_res['zon_reaTRooAir_y'] - 273.15, color='darkorange', linewidth=0.8, label='$T_z$')
    axs[0].set_ylabel('Operative\nTemp ($^\circ$C)')
    axs[0].legend(loc='upper right')
    axs[0].grid(True)

    axs[1].plot(x_time, df_res['zon_weaSta_reaWeaTDryBul_y'] - 273.15, color='green', linewidth=0.8, label='$T_{amb}$')
    axs[1].set_ylabel('Ambient\nTemp ($^\circ$C)')
    axs[1].legend(loc='upper right')
    axs[1].grid(True)

    axs[2].plot(x_time, df_res['fcu_oveTSup_u'] - 273.15, color='red', linewidth=0.8, label='$fcu_oveTSup_u$')
    axs[2].legend(loc='upper right')
    axs[2].grid(True)

    axs[3].plot(x_time, df_res['fcu_oveFan_u'], color='royalblue', linewidth=0.8, label='$fcu_oveFan_u$')
    axs[3].set_xlabel('Time (days)')
    axs[3].legend(loc='upper right')
    axs[3].grid(True)

    plt.tight_layout()
    plt.show()
    
    return df_res

start_in_seconds = start_time_days * 24 * 3600
final_in_seconds = start_in_seconds + (14 * 24 * 3600)

get_and_plot_results(testid=testid, start_time=start_in_seconds, final_time=final_in_seconds)

In [ ]:
ls_rmse = [t.cpu()*30 for t in ls_rmse]

In [ ]:

plt.plot(ls_rmse[:1000])
plt.xlabel('Iteration (number of new data points)')
plt.grid()
plt.ylabel('RMSE (F)')
plt.title('Active learning for GP model RMSE')
plt.show()

In [ ]:
df = pd.DataFrame(np.array(ls_rmse), columns=["RMSE_AL_GP"])
file_name = f"RMSE_UCB_GP_ini{Ne_tr}_{const.ramp}.csv"
save_dir = os.path.join(save_dir, file_name)
df.to_csv(save_dir, index=False)